# m02_l05_retries

Exobean Engineering. Run the cells from top to bottom.

In [ ]:
# Google Colab only: fetches the free code companion, then run the rest.
import io, os, sys, urllib.request, zipfile
URL = "https://github.com/exobean/ai-agents-level1/releases/latest/download/shopassist-companion.zip"
if "google.colab" in sys.modules and not os.path.isdir("shopassist"):
    try:
        data = io.BytesIO(urllib.request.urlopen(URL).read())
    except OSError:  # network blocks the download: upload the zip
        from google.colab import files
        data = next(iter(files.upload()))
    zipfile.ZipFile(data).extractall(".")
    os.chdir(next(d for d in os.listdir(".") if d.endswith("-companion")))
    sys.path.insert(0, os.getcwd())


## Retry

In [ ]:
from shopassist.client import post_json, with_retries
from shopassist.mockserver import MockProvider

body = {"model": "mock", "max_tokens": 200, "system": "Be brief.",
        "messages": [{"role": "user", "content": "Order #1042?"}]}


def log(attempt, status, wait):
    print(f"attempt {attempt} failed with {status};",
          f"waiting {wait:.2f}s")


# The stand-in provider fails three times, then answers.
with MockProvider(fail_with=[429, 500, 503]) as server:
    def call():
        return post_json(server.url, body, "test-key")
    reply = with_retries(call, log=log, seed=7, sleep=lambda s: None)
    calls = len(server.requests)

print("succeeded after", calls, "calls:", reply["content"][0]["text"])
